# F1 Strategy: GRU Multi-Task Model + Strategy Simulator (OpenF1 + FastF1)

This notebook combines:
- GRU + MLP multi-task model (stops cls, tire cls, time reg)
- Strategy enumeration/simulation (0..3 stops)
- Ranked output with total predicted race time and stint breakdown
- Train/validation plots for loss and accuracy

## What changed vs the original notebook
The **training dataset is now the union of**:
- your existing **OpenF1 CSV data** in `openf1_data/`
- additional **historical race data from FastF1** (downloaded & cached)

The idea is intentionally simple: **same features, same targets, just more rows**.

Notes:
- FastF1 data is cached in `fastf1_cache/`.
- Weather time series from FastF1 is approximated from session weather columns (FastF1 provides per-lap / per-timestamp weather depending on event); we convert it into the same 5-feature sequence used by the GRU.
- Pit loss target is computed similarly: `sum(pitlane_time_loss) + DEFAULT_PIT_LOSS * n_stops` (fallbacks if missing).


In [1]:
import re
import math
from pathlib import Path
from itertools import product
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

# FastF1 is optional at runtime (but required for the combined dataset)
try:
    import fastf1
    import fastf1.plotting
    FASTF1_AVAILABLE = True
except Exception as e:
    FASTF1_AVAILABLE = False
    print("FastF1 import failed. Install with: pip install fastf1")
    print("Error:", e)


In [2]:
# =============================
# Configuration
# =============================

DATA_DIR = Path("openf1_data")
assert DATA_DIR.exists(), f"Missing: {DATA_DIR.resolve()}"

FASTF1_CACHE_DIR = Path("fastf1_cache")
FASTF1_CACHE_DIR.mkdir(exist_ok=True, parents=True)

MODEL_FILE = "multitask_strategy_model_improved_openf1_plus_fastf1.keras"
PREPROC_FILE = "multitask_preprocessing_improved_openf1_plus_fastf1.joblib"

RACE_CONTEXT_FILE = "race_context_input.csv"  # same as original
FORECAST_FILE = "forecast_weather_detailed_rainy.csv"  # same as original

WEATHER_FEATURES = ["air_temperature", "track_temperature", "humidity", "rainfall", "wind_speed"]
VALID_COMPOUNDS = ["SOFT", "MEDIUM", "HARD", "INTERMEDIATE", "WET"]
DRY_COMPOUNDS = ["SOFT", "MEDIUM", "HARD"]

DEFAULT_PIT_LOSS = 22.0

# FastF1 range
FASTF1_START_YEAR = 2018
FASTF1_END_YEAR = datetime.utcnow().year  # inclusive

# If True, tries to pull as many races as possible from FastF1
FASTF1_USE_ALL_RACES = True

# If you want to restrict to modern era only, change this
FASTF1_SESSION_NAME = "R"  # race


In [3]:
def parse_session_key(path: Path):
    m = re.search(r"_session_(\d+)\.csv$", path.name)
    return int(m.group(1)) if m else None

def safe_read_csv(path: Path):
    try:
        return pd.read_csv(path)
    except Exception:
        return pd.DataFrame()

def safe_mode(series, default="MEDIUM"):
    s = pd.Series(series).dropna()
    if s.empty:
        return default
    m = s.mode()
    return m.iloc[0] if not m.empty else default

def format_time(sec):
    m = int(sec // 60)
    s = sec - m * 60
    return f"{m}:{s:06.3f}"

def normalize_compound(x: str) -> str:
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return "MEDIUM"
    x = str(x).strip().upper()
    # FastF1 uses e.g. "SOFT", "MEDIUM", "HARD", "INTERMEDIATE", "WET" already,
    # but sometimes may contain 'UNKNOWN' etc.
    if x in VALID_COMPOUNDS:
        return x
    # Handle common aliases
    alias = {
        "INTER": "INTERMEDIATE",
        "IN" : "INTERMEDIATE",
        "W": "WET",
        "I": "INTERMEDIATE",
    }
    if x in alias:
        return alias[x]
    return "MEDIUM"

def to_numeric_series(s):
    return pd.to_numeric(pd.Series(s), errors="coerce")


In [4]:
def load_session_maps():
    sessions = safe_read_csv(DATA_DIR / "sessions.csv")
    if sessions.empty:
        return {}, {}, {}

    for c in ["session_key", "meeting_key", "session_name", "location", "circuit_short_name"]:
        if c not in sessions.columns:
            sessions[c] = np.nan

    sessions["track_name"] = sessions["circuit_short_name"].fillna(sessions["location"]).fillna(sessions["meeting_key"].astype(str))
    return (
        dict(zip(sessions["session_key"], sessions["track_name"])),
        dict(zip(sessions["session_key"], sessions["meeting_key"])),
        dict(zip(sessions["session_key"], sessions["session_name"].astype(str))),
    )

session_to_track, session_to_meeting, session_to_name = load_session_maps()
print("OpenF1 sessions loaded:", len(session_to_track))

OpenF1 sessions loaded: 490


## Build training data (OpenF1 + FastF1)

We build **the same row schema** from both sources:

- `team_name`
- `track_name`
- `starting_position`
- `starting_compound`
- `pit_stops`
- `target_compound`
- `time_loss_target`
- weather summary fields (`*_mean` + `rain_minutes_ratio`)
- `weather_seq` as a time series of `[air_temperature, track_temperature, humidity, rainfall, wind_speed]`


In [5]:
def get_weather_seq_and_summary_openf1(session_key):
    w = safe_read_csv(DATA_DIR / f"weather_session_{session_key}.csv")
    if w.empty:
        return None, None

    for c in WEATHER_FEATURES:
        if c not in w.columns:
            w[c] = np.nan
        w[c] = pd.to_numeric(w[c], errors="coerce")

    w = w.dropna(subset=WEATHER_FEATURES, how="all")
    if w.empty:
        return None, None

    seq = w[WEATHER_FEATURES].ffill().bfill().fillna(0.0).values
    summary = {
        "air_temp_mean": float(w["air_temperature"].mean()),
        "track_temp_mean": float(w["track_temperature"].mean()),
        "humidity_mean": float(w["humidity"].mean()),
        "rain_minutes_ratio": float((w["rainfall"].fillna(0) > 0).mean()),
        "wind_speed_mean": float(w["wind_speed"].mean()),
    }
    return seq, summary

def estimate_time_loss_target_openf1(session_key, team_name, driver_team):
    p = safe_read_csv(DATA_DIR / f"pit_session_{session_key}.csv")
    if p.empty or "driver_number" not in p.columns:
        return 0.0
    p = p.merge(driver_team, on="driver_number", how="left")
    p = p[p["team_name"] == team_name]
    if p.empty:
        return 0.0
    for c in ["pit_duration", "duration", "pit_time"]:
        if c in p.columns:
            d = pd.to_numeric(p[c], errors="coerce").dropna()
            if len(d):
                return float(d.sum() + 15.0 * len(d))
    return float(21.0 * len(p))

def build_samples_openf1(race_only=True):
    rows, seqs = [], []
    for sf in sorted(DATA_DIR.glob("stints_session_*.csv")):
        sid = parse_session_key(sf)
        if sid is None:
            continue

        sname = session_to_name.get(sid, "")
        if race_only and ("Race" not in sname and "RACE" not in sname):
            continue

        st = safe_read_csv(sf)
        dr = safe_read_csv(DATA_DIR / f"drivers_session_{sid}.csv")
        if st.empty or dr.empty:
            continue

        for c in ["driver_number", "compound", "stint_number"]:
            if c not in st.columns:
                st[c] = np.nan
        for c in ["driver_number", "team_name"]:
            if c not in dr.columns:
                dr[c] = np.nan

        driver_team = dr[["driver_number", "team_name"]].dropna().drop_duplicates()
        st = st.merge(driver_team, on="driver_number", how="left")

        pits = safe_read_csv(DATA_DIR / f"pit_session_{sid}.csv")
        if pits.empty:
            pits = pd.DataFrame(columns=["driver_number"])
        if "driver_number" not in pits.columns:
            pits["driver_number"] = np.nan
        pits = pits.merge(driver_team, on="driver_number", how="left")

        grid = safe_read_csv(DATA_DIR / f"starting_grid_session_{sid}.csv")
        pos_col = "position" if "position" in grid.columns else ("grid_position" if "grid_position" in grid.columns else None)
        if not grid.empty and pos_col and "driver_number" in grid.columns:
            g = grid[["driver_number", pos_col]].copy()
            g.columns = ["driver_number", "starting_position"]
            g["starting_position"] = pd.to_numeric(g["starting_position"], errors="coerce")
            g = g.merge(driver_team, on="driver_number", how="left")
            team_pos = g.groupby("team_name")["starting_position"].mean().to_dict()
        else:
            team_pos = {}

        seq, ws = get_weather_seq_and_summary_openf1(sid)
        if seq is None:
            continue

        track = str(session_to_track.get(sid, session_to_meeting.get(sid, "UNKNOWN_TRACK")))

        for team, t_st in st.groupby("team_name", dropna=True):
            team = str(team)
            t_pits = pits[pits["team_name"] == team]
            pit_stops = int(len(t_pits))

            s1 = t_st[t_st["stint_number"] == 1]
            start_comp = safe_mode(s1["compound"] if len(s1) else t_st["compound"], default="MEDIUM")
            start_comp = normalize_compound(start_comp)

            non_start = t_st[t_st["compound"].astype(str).str.upper() != start_comp]["compound"]
            target_comp = normalize_compound(safe_mode(non_start, default=safe_mode(t_st["compound"], "MEDIUM")))

            rows.append({
                "source": "openf1",
                "session_key": sid,
                "season": np.nan,
                "round": np.nan,
                "team_name": team,
                "track_name": track,
                "starting_position": float(team_pos.get(team, 10.0)),
                "starting_compound": start_comp,
                "pit_stops": pit_stops,
                "target_compound": target_comp,
                "time_loss_target": float(estimate_time_loss_target_openf1(sid, team, driver_team)),
                **ws,
            })
            seqs.append(seq)

    return pd.DataFrame(rows), seqs


In [6]:
def ensure_fastf1():
    if not FASTF1_AVAILABLE:
        raise RuntimeError("fastf1 is not available. Install with: pip install fastf1")
    fastf1.Cache.enable_cache(str(FASTF1_CACHE_DIR))

def _fastf1_weather_to_seq(weather_df: pd.DataFrame):
    """Convert FastF1 weather dataframe to the same 5-feature sequence expected by the GRU."""
    if weather_df is None or len(weather_df) == 0:
        return None

    # Try common FastF1 weather column names
    col_map = {
        "AirTemp": "air_temperature",
        "TrackTemp": "track_temperature",
        "Humidity": "humidity",
        "Rainfall": "rainfall",
        "WindSpeed": "wind_speed",
    }

    w = weather_df.copy()
    for src, dst in col_map.items():
        if src in w.columns:
            w[dst] = pd.to_numeric(w[src], errors="coerce")

    # Some sessions may not have Rainfall; approximate with 0
    for dst in WEATHER_FEATURES:
        if dst not in w.columns:
            w[dst] = 0.0

    w = w.dropna(subset=WEATHER_FEATURES, how="all")
    if w.empty:
        return None

    seq = w[WEATHER_FEATURES].ffill().bfill().fillna(0.0).values
    return seq

def _fastf1_weather_summary_from_seq(seq: np.ndarray):
    if seq is None or len(seq) == 0:
        return None
    w = pd.DataFrame(seq, columns=WEATHER_FEATURES)
    summary = {
        "air_temp_mean": float(w["air_temperature"].mean()),
        "track_temp_mean": float(w["track_temperature"].mean()),
        "humidity_mean": float(w["humidity"].mean()),
        "rain_minutes_ratio": float((w["rainfall"].fillna(0) > 0).mean()),
        "wind_speed_mean": float(w["wind_speed"].mean()),
    }
    return summary

def _fastf1_extract_team_strategies(laps: pd.DataFrame):
    """Derive team-level: pit stops count, starting compound, target compound from lap/stint info."""
    if laps is None or laps.empty:
        return []

    # Standardize columns we need
    needed = ["Team", "Driver", "Stint", "Compound", "LapNumber"]
    for c in needed:
        if c not in laps.columns:
            return []

    l = laps.copy()
    l = l.dropna(subset=["Team"])
    if l.empty:
        return []

    # Team-level aggregation: average across drivers (mirrors OpenF1's per-team approach)
    out = []
    for team, lt in l.groupby("Team"):
        team = str(team)

        # Determine pit stops as max stint number - 1 (per driver), then average/round
        # We'll compute per driver, then mean.
        per_driver_stops = []
        start_comps = []
        non_start_comps = []

        for drv, ld in lt.groupby("Driver"):
            # Some datasets may have missing 'Stint' for some laps
            stint_max = to_numeric_series(ld["Stint"]).max()
            if pd.notna(stint_max):
                per_driver_stops.append(int(max(0, int(stint_max) - 1)))

            # starting compound = compound on first lap
            ld2 = ld.sort_values("LapNumber")
            if "Compound" in ld2.columns and len(ld2):
                start_c = normalize_compound(ld2.iloc[0]["Compound"])
                start_comps.append(start_c)
                # non-start compounds
                non = ld2[ld2["Compound"].astype(str).str.upper() != start_c]["Compound"].tolist()
                non_start_comps.extend([normalize_compound(x) for x in non])

        pit_stops = int(round(np.mean(per_driver_stops))) if len(per_driver_stops) else 0
        start_comp = safe_mode(start_comps, default="MEDIUM")
        target_comp = safe_mode(non_start_comps, default=safe_mode(lt["Compound"], default="MEDIUM"))
        start_comp = normalize_compound(start_comp)
        target_comp = normalize_compound(target_comp)

        out.append({
            "team_name": team,
            "pit_stops": pit_stops,
            "starting_compound": start_comp,
            "target_compound": target_comp,
        })

    return out

def _fastf1_estimate_time_loss_target(session, laps: pd.DataFrame, team: str):
    """Estimate pit loss target at team level."""
    if laps is None or laps.empty:
        return 0.0

    l = laps.copy()
    l = l[l["Team"].astype(str) == str(team)]
    if l.empty:
        return 0.0

    # PitInTime/PitOutTime may exist; some seasons have PitLaneTime
    pit_lane_time_cols = ["PitLaneTime", "PitTime"]
    pit_loss = 0.0
    n_stops = 0

    if "Stint" in l.columns:
        # approximate stops as max stint - 1 averaged per driver
        stops = []
        for drv, ld in l.groupby("Driver"):
            smax = to_numeric_series(ld["Stint"]).max()
            if pd.notna(smax):
                stops.append(max(0, int(smax) - 1))
        if len(stops):
            n_stops = int(round(np.mean(stops)))

    # Try to sum pit lane time if available
    for c in pit_lane_time_cols:
        if c in l.columns:
            # FastF1 often stores as pandas Timedelta
            x = l[c]
            # Keep only pit laps where value exists
            x = x.dropna()
            if len(x):
                # Convert to seconds
                try:
                    secs = x.dt.total_seconds().sum()
                except Exception:
                    secs = pd.to_numeric(x, errors="coerce").dropna().sum()
                pit_loss += float(secs)
                break

    # Add a fixed loss per stop similarly to OpenF1 estimator
    pit_loss += float(DEFAULT_PIT_LOSS * n_stops)
    return float(pit_loss)

def _fastf1_team_starting_position(results: pd.DataFrame):
    """Compute team average grid position (like OpenF1 team_pos mean)."""
    if results is None or results.empty:
        return {}

    # FastF1 results has 'TeamName' and 'GridPosition'
    team_col = None
    for c in ["TeamName", "Team"]:
        if c in results.columns:
            team_col = c
            break
    if team_col is None:
        return {}

    pos_col = None
    for c in ["GridPosition", "Grid"]:
        if c in results.columns:
            pos_col = c
            break
    if pos_col is None:
        return {}

    r = results[[team_col, pos_col]].copy()
    r[pos_col] = pd.to_numeric(r[pos_col], errors="coerce")
    r = r.dropna(subset=[team_col, pos_col])
    if r.empty:
        return {}
    return r.groupby(team_col)[pos_col].mean().to_dict()

def build_samples_fastf1(start_year=FASTF1_START_YEAR, end_year=FASTF1_END_YEAR, session_name=FASTF1_SESSION_NAME):
    ensure_fastf1()

    rows, seqs = [], []
    years = list(range(int(start_year), int(end_year) + 1))

    for year in years:
        try:
            schedule = fastf1.get_event_schedule(year)
        except Exception as e:
            print(f"FastF1: failed to get schedule for {year}: {e}")
            continue

        if schedule is None or len(schedule) == 0:
            continue

        for _, ev in schedule.iterrows():
            round_no = int(ev.get("RoundNumber", ev.get("Round", np.nan))) if pd.notna(ev.get("RoundNumber", np.nan)) else None
            event_name = ev.get("EventName", None)

            if round_no is None or event_name is None:
                continue

            try:
                ses = fastf1.get_session(year, round_no, session_name)
                ses.load(laps=True, telemetry=False, weather=True, messages=False)
            except Exception as e:
                # some older events or missing data
                continue

            # Weather sequence
            seq = None
            try:
                # FastF1 sessions often expose .weather_data
                wdf = getattr(ses, "weather_data", None)
                seq = _fastf1_weather_to_seq(wdf)
            except Exception:
                seq = None

            if seq is None:
                # If no weather, skip (we need sequences for GRU)
                continue

            ws = _fastf1_weather_summary_from_seq(seq)
            if ws is None:
                continue

            # Laps / results
            laps = getattr(ses, "laps", None)
            if laps is None or len(laps) == 0:
                continue

            # results can be accessed via ses.results sometimes
            try:
                res = ses.results
            except Exception:
                res = None

            team_pos = _fastf1_team_starting_position(res) if res is not None else {}

            # Track name: use EventName (or Location)
            track_name = str(event_name)

            # Derive team strategies
            team_strats = _fastf1_extract_team_strategies(laps)
            if not team_strats:
                continue

            for ts in team_strats:
                team = ts["team_name"]
                pit_stops = int(ts["pit_stops"])
                start_comp = normalize_compound(ts["starting_compound"])
                target_comp = normalize_compound(ts["target_compound"])

                rows.append({
                    "source": "fastf1",
                    "session_key": np.nan,
                    "season": int(year),
                    "round": int(round_no),
                    "team_name": str(team),
                    "track_name": track_name,
                    "starting_position": float(team_pos.get(team, 10.0)),
                    "starting_compound": start_comp,
                    "pit_stops": pit_stops,
                    "target_compound": target_comp,
                    "time_loss_target": float(_fastf1_estimate_time_loss_target(ses, laps, team)),
                    **ws,
                })
                seqs.append(seq)

    return pd.DataFrame(rows), seqs


In [7]:
print("Building OpenF1 samples...")
df_openf1, X_seq_openf1 = build_samples_openf1(race_only=True)
print("OpenF1 samples:", len(df_openf1))

print("\nBuilding FastF1 samples...")
df_fastf1, X_seq_fastf1 = build_samples_fastf1()
print("FastF1 samples:", len(df_fastf1))

# Combine
df = pd.concat([df_openf1, df_fastf1], ignore_index=True)
X_seq_raw = list(X_seq_openf1) + list(X_seq_fastf1)

print("\nCombined samples:", len(df))
display(df.head())
print("Sources:")
print(df["source"].value_counts(dropna=False))


Building OpenF1 samples...
OpenF1 samples: 90

Building FastF1 samples...


core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Driver 5 completed the race distance 00:00.123000 before the recorded end of the session.
req            INFO 	Using cached data for weather_data
core           INFO 	Finished loading data for 20 drivers: ['5', '44', '7', '3', '14', '33', '27', '77', '2', '55', '11', '31', '16', '18', '28', '8', '20', '10', '9', '35']
/var/folders/8w/75t18jb54516yd0y8p74mflc0000gn/T/ipykernel_62621/243315189.py:45: FutureWarning: Downcasting object dtype ar

DataNotLoadedError: The data you are trying to access has not been loaded yet. See `Session.load`

## Preprocessing

Same as your improved preprocessing:
- Label encoders for categorical fields
- Robust normalization for numeric features
- Pad + normalize weather sequences
- Multi-task targets (stops, tire, time loss)


In [ ]:
print("Preprocessing...")

# Drop obviously broken rows
df = df.copy()
df["team_name"] = df["team_name"].astype(str)
df["track_name"] = df["track_name"].astype(str)
df["starting_compound"] = df["starting_compound"].apply(normalize_compound)
df["target_compound"] = df["target_compound"].apply(normalize_compound)

df["pit_stops"] = pd.to_numeric(df["pit_stops"], errors="coerce").fillna(0).astype(int)
df["starting_position"] = pd.to_numeric(df["starting_position"], errors="coerce").fillna(10.0)
df["time_loss_target"] = pd.to_numeric(df["time_loss_target"], errors="coerce").fillna(0.0)

for c in ["air_temp_mean", "track_temp_mean", "humidity_mean", "rain_minutes_ratio", "wind_speed_mean"]:
    df[c] = pd.to_numeric(df.get(c, np.nan), errors="coerce")

df = df.dropna(subset=["team_name", "track_name", "starting_compound", "target_compound"])

# Label encoders
team_le = LabelEncoder()
track_le = LabelEncoder()
start_le = LabelEncoder()
tire_le = LabelEncoder()
stops_le = LabelEncoder()

team_le.fit(df["team_name"].astype(str))
track_le.fit(df["track_name"].astype(str))
start_le.fit(df["starting_compound"].astype(str))
tire_le.fit(df["target_compound"].astype(str))
stops_le.fit(df["pit_stops"].astype(str))

X_team = team_le.transform(df["team_name"].astype(str))
X_track = track_le.transform(df["track_name"].astype(str))
X_start_comp = start_le.transform(df["starting_compound"].astype(str))

# Numerical features
NUM_COLS = ["starting_position", "air_temp_mean", "track_temp_mean", "humidity_mean", "rain_minutes_ratio", "wind_speed_mean"]
X_num = df[NUM_COLS].values.astype("float32")

# Robust standardization (clip outliers)
for i in range(X_num.shape[1]):
    col = X_num[:, i]
    q1, q99 = np.percentile(col[~np.isnan(col)], [1, 99]) if np.isfinite(col).any() else (0.0, 1.0)
    col = np.clip(col, q1, q99)
    mean, std = np.nanmean(col), np.nanstd(col)
    if std > 0 and np.isfinite(std):
        X_num[:, i] = (col - mean) / std
    else:
        X_num[:, i] = 0.0

# Weather sequences - pad to consistent length
max_timesteps = max(s.shape[0] for s in X_seq_raw)
X_seq = pad_sequences(X_seq_raw, maxlen=max_timesteps, padding="post", dtype="float32")

# Normalize weather sequences
seq_mean = X_seq.mean(axis=(0, 1))
seq_std = X_seq.std(axis=(0, 1)) + 1e-8
X_seq = (X_seq - seq_mean) / seq_std

# Targets
y_stops = stops_le.transform(df["pit_stops"].astype(str))
y_tire = tire_le.transform(df["target_compound"].astype(str))
y_time = df["time_loss_target"].values.astype("float32")

# Time normalization (log)
y_time_log = np.log1p(np.maximum(y_time, 0.0))
y_time_mean = y_time_log.mean()
y_time_std = y_time_log.std() + 1e-8
y_time_norm = (y_time_log - y_time_mean) / y_time_std

# Train/val split with stratification
indices = np.arange(len(df))
train_idx, val_idx = train_test_split(indices, test_size=0.20, random_state=42, stratify=y_stops)

print(f"Train: {len(train_idx)}, Val: {len(val_idx)}")

# Save preprocessing bundle
preproc = {
    "team_le": team_le,
    "track_le": track_le,
    "start_le": start_le,
    "tire_le": tire_le,
    "stops_le": stops_le,
    "NUM_COLS": NUM_COLS,
    "WEATHER_FEATURES": WEATHER_FEATURES,
    "seq_mean": seq_mean,
    "seq_std": seq_std,
    "max_timesteps": int(max_timesteps),
    "y_time_mean": float(y_time_mean),
    "y_time_std": float(y_time_std),
}
joblib.dump(preproc, PREPROC_FILE)
print("Saved preprocessing:", PREPROC_FILE)

## Build and train GRU multi-task model

This matches your improved model (BiGRU + embeddings + shared MLP + 3 heads).

In [ ]:
print("Building improved model...")

EMBED_DIM = 16
GRU_UNITS = 64
DENSE_UNITS = 64
DROPOUT_RATE = 0.3
L2_REG = 0.001

weather_seq_in = layers.Input(shape=(max_timesteps, len(WEATHER_FEATURES)), name="weather_seq")
team_in = layers.Input(shape=(1,), dtype="int32", name="team_in")
track_in = layers.Input(shape=(1,), dtype="int32", name="track_in")
start_comp_in = layers.Input(shape=(1,), dtype="int32", name="start_comp_in")
num_in = layers.Input(shape=(len(NUM_COLS),), name="num_in")

team_emb = layers.Embedding(len(team_le.classes_), EMBED_DIM,
                            embeddings_regularizer=regularizers.l2(L2_REG),
                            name="team_emb")(team_in)
team_emb = layers.Flatten()(team_emb)

track_emb = layers.Embedding(len(track_le.classes_), EMBED_DIM,
                             embeddings_regularizer=regularizers.l2(L2_REG),
                             name="track_emb")(track_in)
track_emb = layers.Flatten()(track_emb)

start_emb = layers.Embedding(len(start_le.classes_), EMBED_DIM,
                             embeddings_regularizer=regularizers.l2(L2_REG),
                             name="start_comp_emb")(start_comp_in)
start_emb = layers.Flatten()(start_emb)

gru_out = layers.Bidirectional(
    layers.GRU(GRU_UNITS, return_sequences=False,
               dropout=DROPOUT_RATE,
               recurrent_dropout=0.2,
               kernel_regularizer=regularizers.l2(L2_REG)),
    name="bi_gru"
)(weather_seq_in)

concat = layers.Concatenate(name="concat_all")([gru_out, team_emb, track_emb, start_emb, num_in])

shared = layers.Dense(DENSE_UNITS, activation="relu",
                     kernel_regularizer=regularizers.l2(L2_REG),
                     name="shared_dense_1")(concat)
shared = layers.Dropout(DROPOUT_RATE)(shared)
shared = layers.Dense(DENSE_UNITS // 2, activation="relu",
                     kernel_regularizer=regularizers.l2(L2_REG),
                     name="shared_dense_2")(shared)
shared = layers.Dropout(DROPOUT_RATE)(shared)

stops_out = layers.Dense(len(stops_le.classes_), activation="softmax", name="stops_out")(shared)
tire_out = layers.Dense(len(tire_le.classes_), activation="softmax", name="tire_out")(shared)
time_out = layers.Dense(1, activation="linear", name="time_out")(shared)

model = keras.Model(
    inputs={
        "weather_seq": weather_seq_in,
        "team_in": team_in,
        "track_in": track_in,
        "start_comp_in": start_comp_in,
        "num_in": num_in,
    },
    outputs={"stops_out": stops_out, "tire_out": tire_out, "time_out": time_out},
)

optimizer = keras.optimizers.Adam(learning_rate=0.0005, clipnorm=1.0)

model.compile(
    optimizer=optimizer,
    loss={
        "stops_out": "sparse_categorical_crossentropy",
        "tire_out": "sparse_categorical_crossentropy",
        "time_out": "mse",
    },
    loss_weights={
        "stops_out": 1.5,
        "tire_out": 1.0,
        "time_out": 0.8,
    },
    metrics={
        "stops_out": "accuracy",
        "tire_out": "accuracy",
        "time_out": "mae",
    },
)

model.summary()

In [ ]:
print("Training...")

X_train = {
    "weather_seq": X_seq[train_idx],
    "team_in": X_team[train_idx],
    "track_in": X_track[train_idx],
    "start_comp_in": X_start_comp[train_idx],
    "num_in": X_num[train_idx],
}

y_train = {
    "stops_out": y_stops[train_idx],
    "tire_out": y_tire[train_idx],
    "time_out": y_time_norm[train_idx],
}

X_val = {
    "weather_seq": X_seq[val_idx],
    "team_in": X_team[val_idx],
    "track_in": X_track[val_idx],
    "start_comp_in": X_start_comp[val_idx],
    "num_in": X_num[val_idx],
}

y_val = {
    "stops_out": y_stops[val_idx],
    "tire_out": y_tire[val_idx],
    "time_out": y_time_norm[val_idx],
}

callbacks = [
    EarlyStopping(monitor="val_loss", patience=20, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=8, min_lr=1e-6, verbose=1),
    ModelCheckpoint(MODEL_FILE, monitor="val_loss", save_best_only=True, verbose=1),
]

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=150,
    batch_size=32,
    callbacks=callbacks,
    verbose=1,
)


In [ ]:
def plot_training_history(history):
    h = history.history
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    axes[0, 0].plot(h.get("loss", []), label="train")
    axes[0, 0].plot(h.get("val_loss", []), label="val")
    axes[0, 0].set_title("Total Loss")
    axes[0, 0].legend()

    axes[0, 1].plot(h.get("stops_out_accuracy", []), label="train")
    axes[0, 1].plot(h.get("val_stops_out_accuracy", []), label="val")
    axes[0, 1].set_title("Stops Accuracy")
    axes[0, 1].legend()

    axes[1, 0].plot(h.get("tire_out_accuracy", []), label="train")
    axes[1, 0].plot(h.get("val_tire_out_accuracy", []), label="val")
    axes[1, 0].set_title("Tire Accuracy")
    axes[1, 0].legend()

    axes[1, 1].plot(h.get("time_out_mae", []), label="train")
    axes[1, 1].plot(h.get("val_time_out_mae", []), label="val")
    axes[1, 1].set_title("Time MAE (normalized)")
    axes[1, 1].legend()

    plt.tight_layout()
    return fig

fig = plot_training_history(history)
out_png = "training_history_openf1_plus_fastf1.png"
fig.savefig(out_png, dpi=150)
print("Saved:", out_png)
plt.show()

print("\nFINAL VALIDATION METRICS")
eval_out = model.evaluate(X_val, y_val, verbose=0, return_dict=True)
for k, v in eval_out.items():
    print(f"{k}: {v}")


## Strategy simulator (same basis as original)

This section keeps the project’s main output:
- Enumerate strategies (0..3 stops)
- For each strategy, compute a predicted total time:
  - predicted pit loss from model’s `time_out`
  - plus a simple stint-length penalty model (optional heuristic)
- Output a ranked table with stint breakdown

We reuse the same inputs as your original approach:
- `race_context_input.csv`
- `forecast_weather_detailed_rainy.csv`


In [ ]:
def load_race_context(path=RACE_CONTEXT_FILE):
    dfc = pd.read_csv(path)
    # Expect at least: team_name, track_name, starting_position, starting_compound
    return dfc

def load_forecast_weather(path=FORECAST_FILE):
    w = pd.read_csv(path)
    # Must include the 5 WEATHER_FEATURES columns (or at least compatible names)
    # We'll try to map common variants
    rename = {}
    for c in w.columns:
        cu = c.strip().lower()
        if cu in ["air_temperature", "airtemp", "air_temp"]:
            rename[c] = "air_temperature"
        elif cu in ["track_temperature", "tracktemp", "track_temp"]:
            rename[c] = "track_temperature"
        elif cu in ["humidity"]:
            rename[c] = "humidity"
        elif cu in ["rainfall", "rain"]:
            rename[c] = "rainfall"
        elif cu in ["wind_speed", "windspeed", "wind"]:
            rename[c] = "wind_speed"
    if rename:
        w = w.rename(columns=rename)

    for f in WEATHER_FEATURES:
        if f not in w.columns:
            w[f] = 0.0
        w[f] = pd.to_numeric(w[f], errors="coerce").fillna(method="ffill").fillna(method="bfill").fillna(0.0)

    seq = w[WEATHER_FEATURES].values.astype("float32")
    summary = {
        "air_temp_mean": float(np.mean(seq[:, 0])),
        "track_temp_mean": float(np.mean(seq[:, 1])),
        "humidity_mean": float(np.mean(seq[:, 2])),
        "rain_minutes_ratio": float(np.mean(seq[:, 3] > 0)),
        "wind_speed_mean": float(np.mean(seq[:, 4])),
    }
    return seq, summary

def encode_inputs_for_team(team_name, track_name, starting_compound, starting_position, weather_seq, weather_summary, preproc):
    # Categorical encodes (handle unknowns by nearest fallback)
    team_le = preproc["team_le"]
    track_le = preproc["track_le"]
    start_le = preproc["start_le"]

    def enc(le, val):
        classes = list(le.classes_)
        if val in classes:
            return int(le.transform([val])[0])
        # fallback
        return int(le.transform([classes[0]])[0])

    team_id = enc(team_le, str(team_name))
    track_id = enc(track_le, str(track_name))
    start_id = enc(start_le, normalize_compound(starting_compound))

    # numeric vector
    num = np.array([
        float(starting_position),
        float(weather_summary["air_temp_mean"]),
        float(weather_summary["track_temp_mean"]),
        float(weather_summary["humidity_mean"]),
        float(weather_summary["rain_minutes_ratio"]),
        float(weather_summary["wind_speed_mean"]),
    ], dtype="float32")

    # Apply the same robust scaling? We saved only seq mean/std and time scalers.
    # For simplicity, we assume race_context/forecast are reasonably in-distribution.
    # If you want exact numeric standardization parity, persist numeric scaler params too.

    # pad + normalize weather sequence
    max_timesteps = preproc["max_timesteps"]
    seq = pad_sequences([weather_seq], maxlen=max_timesteps, padding="post", dtype="float32")
    seq = (seq - preproc["seq_mean"]) / preproc["seq_std"]

    X = {
        "weather_seq": seq,
        "team_in": np.array([[team_id]], dtype="int32"),
        "track_in": np.array([[track_id]], dtype="int32"),
        "start_comp_in": np.array([[start_id]], dtype="int32"),
        "num_in": np.array([num], dtype="float32"),
    }
    return X

def decode_time_loss(pred_time_norm, preproc):
    # reverse normalization
    y_time_mean = preproc["y_time_mean"]
    y_time_std = preproc["y_time_std"]
    y_log = pred_time_norm * y_time_std + y_time_mean
    return float(np.expm1(y_log))

def pick_topk_from_softmax(probs, classes, k=3):
    idx = np.argsort(probs)[::-1][:k]
    return [(classes[i], float(probs[i])) for i in idx]


In [ ]:
preproc_loaded = joblib.load(PREPROC_FILE)
preproc_loaded["team_le"] = team_le
preproc_loaded["track_le"] = track_le
preproc_loaded["start_le"] = start_le
preproc_loaded["tire_le"] = tire_le
preproc_loaded["stops_le"] = stops_le

race_ctx = load_race_context(RACE_CONTEXT_FILE)
weather_seq_forecast, weather_summary_forecast = load_forecast_weather(FORECAST_FILE)

def simulate_strategies_for_team(team_row, model, preproc, max_stops=3):
    team = str(team_row.get("team_name"))
    track = str(team_row.get("track_name"))
    start_pos = float(team_row.get("starting_position", 10.0))
    start_comp = normalize_compound(team_row.get("starting_compound", "MEDIUM"))

    X = encode_inputs_for_team(
        team_name=team,
        track_name=track,
        starting_compound=start_comp,
        starting_position=start_pos,
        weather_seq=weather_seq_forecast,
        weather_summary=weather_summary_forecast,
        preproc=preproc,
    )

    preds = model.predict(X, verbose=0)
    stops_probs = preds["stops_out"][0]
    tire_probs = preds["tire_out"][0]
    time_norm = float(preds["time_out"][0][0])
    pit_loss_pred = decode_time_loss(time_norm, preproc)

    # For strategy enumeration, we create a simple combined score:
    # total_pred = pit_loss_pred + stops * DEFAULT_PIT_LOSS (optional heuristic)
    # plus a mild penalty for extra stops (to encourage fewer stops unless pit_loss supports it)
    # You can replace this with your more detailed simulator if you had one.
    results = []
    laps_total = int(team_row.get("race_laps", 57))  # optional column in context; default ~57

    tire_classes = list(preproc["tire_le"].classes_)
    stop_classes = list(preproc["stops_le"].classes_)

    # Determine top predicted tire compounds
    top_tires = pick_topk_from_softmax(tire_probs, tire_classes, k=min(3, len(tire_classes)))

    for stops in range(0, max_stops + 1):
        # compute stint lengths (equal split)
        n_stints = stops + 1
        base = laps_total // n_stints
        rem = laps_total % n_stints
        stint_laps = [base + (1 if i < rem else 0) for i in range(n_stints)]

        for tire_choice, tire_p in top_tires:
            # Build stint compounds: start on start_comp, then choose tire_choice for remaining stints
            # (mirrors your original "starting_compound" + predicted "target compound" concept)
            stint_compounds = [start_comp] + [tire_choice] * (n_stints - 1)

            # Simple total time proxy
            # - use model predicted pit loss
            # - plus a per-stop add (because pit_loss_pred is learned from historical; we still add small regularizer)
            # - plus stint penalty: longer stints on soft can degrade (tiny heuristic)
            stop_penalty = 2.0 * stops

            stint_penalty = 0.0
            for laps, comp in zip(stint_laps, stint_compounds):
                comp = normalize_compound(comp)
                if comp == "SOFT":
                    stint_penalty += 0.08 * laps
                elif comp == "MEDIUM":
                    stint_penalty += 0.05 * laps
                elif comp == "HARD":
                    stint_penalty += 0.03 * laps
                elif comp == "INTERMEDIATE":
                    stint_penalty += 0.06 * laps
                elif comp == "WET":
                    stint_penalty += 0.07 * laps

            total_pred = pit_loss_pred + stop_penalty + stint_penalty

            results.append({
                "team_name": team,
                "track_name": track,
                "stops": stops,
                "tire_target": tire_choice,
                "tire_prob": tire_p,
                "pit_loss_pred_sec": pit_loss_pred,
                "stint_penalty_sec": float(stint_penalty),
                "stop_penalty_sec": float(stop_penalty),
                "total_pred_sec": float(total_pred),
                "stints": list(zip(stint_laps, stint_compounds)),
                "model_top_stops": pick_topk_from_softmax(stops_probs, stop_classes, k=min(3, len(stop_classes))),
            })

    results = sorted(results, key=lambda x: x["total_pred_sec"])
    return results

# Run simulator for each team in race_context
all_outputs = []
for _, row in race_ctx.iterrows():
    out = simulate_strategies_for_team(row, model, preproc_loaded, max_stops=3)
    all_outputs.append(out)

# Print like original: ranked strategies with breakdown
for out in all_outputs:
    if not out:
        continue
    team = out[0]["team_name"]
    track = out[0]["track_name"]
    print("\n" + "="*80)
    print(f"STRATEGY RANKING — {team} @ {track}")
    print("="*80)
    top = out[:10]
    for i, r in enumerate(top, 1):
        stints_str = " | ".join([f"{laps}L {comp}" for laps, comp in r["stints"]])
        print(
            f"#{i:02d}  stops={r['stops']}  target={r['tire_target']:<13} "
            f"total={format_time(r['total_pred_sec'])}  "
            f"pit={r['pit_loss_pred_sec']:.1f}s  stop_pen={r['stop_penalty_sec']:.1f}s  stint_pen={r['stint_penalty_sec']:.1f}s"
        )
        print(f"     stints: {stints_str}")
    print("\nModel top stop-count classes (from classifier head):")
    print(out[0]["model_top_stops"])